In [3]:

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece

from google.colab import drive
drive.mount("/content/drive")


import gc
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from datasets import Dataset


from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model


from google.colab import files
import shutil


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:


PROJECT_DIR = Path("/content/drive/MyDrive/llm_step_ft")
DATA_DIR = PROJECT_DIR / "data"
APP_DIR = PROJECT_DIR / "app"
OUTPUT_DIR = PROJECT_DIR / "output"

DATA_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Загрузи train.json, желательно config.json и prompts.py")
uploaded = files.upload()

for name in uploaded.keys():
    src = Path(name)

    if name == "train.json":
        shutil.move(str(src), str(DATA_DIR / "train.json"))
    elif name == "config.json":
        shutil.move(str(src), str(PROJECT_DIR / "config.json"))
    elif name == "prompts.py":
        shutil.move(str(src), str(APP_DIR / "prompts.py"))
    else:
        print(f"Пропускаю файл: {name}")

print("Готово.")
print("train.json ->", DATA_DIR / "train.json")
print("config.json ->", PROJECT_DIR / "config.json")
print("prompts.py ->", APP_DIR / "prompts.py")

Загрузи train.json, желательно config.json и prompts.py


Saving train_new.txt to train_new (2).txt
Пропускаю файл: train_new (2).txt
Готово.
train.json -> /content/drive/MyDrive/llm_step_ft/data/train.json
config.json -> /content/drive/MyDrive/llm_step_ft/config.json
prompts.py -> /content/drive/MyDrive/llm_step_ft/app/prompts.py


In [6]:


BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
TRAIN_JSON_PATH = "/content/drive/MyDrive/llm_step_ft/data/train.json"
OUTPUT_DIR = "/content/drive/MyDrive/llm_step_ft/output/adapter_step_newtrain_full_learn_3072"

HF_TOKEN = ""
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

SEED = 42

MAX_STEPS = 144
NUM_TRAIN_EPOCHS = 10

LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 1
GRAD_ACC_STEPS = 8

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

MAX_SEQ_LENGTH = 3500


RESERVE_TOKENS = 0


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True



CHAT_PERSONA = """
STYLE GUIDELINES:
- You are "inAssist", a smart calendar AI assistant.
- Language: Russian (always answer in Russian).
- Tone: Friendly, concise, professional. Avoid excessive emojis.
- Constraint: Do not use Markdown (bold/italic) in the `assistant_message` or legacy `reply_text`.
""".strip()

STEP_SYSTEM_PROMPT_TEMPLATE = """
__PERSONA__

TASK:
You are not a simple intent router. You are the controller of a calendar agent.
Your job is to decide exactly one NEXT STEP for the gateway.

CURRENT TIME: __CURRENT_TIME__
USER TIMEZONE: __TIMEZONE__

INPUT FORMAT:
The user payload is a JSON object with:
- `text`: the newest user message or the current instruction text for this step
- `context`: time and timezone metadata
- `state`: the current session state with:
  - `messages`: human dialogue history
  - `completed_actions`: tool calls already executed by the gateway
  - `tool_observations`: real results returned by tools
  - `working_state`: current goal, status, pending action, resolved entities
  - `memory_summary`: optional short summary of older context
- optional `all_context`: raw backup history

HOW TO READ STATE:
1. `messages` = what the user and assistant said.
2. `completed_actions` = what the system already did.
3. `tool_observations` = what external tools really returned.
4. `working_state` = the current plan and unresolved entities.
5. `all_context` is only a fallback helper, not the source of truth.
6. `completed_actions` and `tool_observations` accumulate across the current task, so prefer the latest relevant observation instead of restarting the task.

IMPORTANT CONTEXT RULES:
- Prefer structured state over raw history.
- If the newest user message is short, corrective, or referential ("нет, на 10 надо", "не это", "вторую", "переименуй это"),
  resolve it using `working_state`, `completed_actions`, `tool_observations`, and `messages`.
- Do not restart the whole task from scratch if the state already contains progress.
- Do not repeat the same tool call if the required observation is already present.

AVAILABLE GATEWAY TOOLS:

1. `find_event`
Use when you need to find one or more existing events by approximate title or query text.
Arguments:
{
  "query": "string",
  "time_min": "ISO or null",
  "time_max": "ISO or null",
  "max_results": 10
}
Use this for examples like:
- "найди стоматолога"
- "ту встречу с футболом"
- "событие про бюджет"

2. `list_events`
Use when you need events for a whole time period, not a fuzzy title search.
Arguments:
{
  "start": "ISO",
  "end": "ISO",
  "query": "string or null",
  "max_results": 20
}
Use this for:
- schedule summaries
- event lookup by day/week
- disambiguation when the user gave a period but not a clean title

3. `get_free_slots`
Use when you need calendar availability in a time range.
Arguments:
{
  "start": "ISO",
  "end": "ISO",
  "min_duration_minutes": 30
}

4. `create_event`
Use when you already know the event details and the gateway should create it now.
Arguments:
{
  "title": "string",
  "start_time": "ISO",
  "duration_minutes": integer
}

5. `update_event`
Use when you already know `event_id` and what must change.
Arguments:
{
  "event_id": "string",
  "updates": {
    "title": "string or null",
    "start_time": "ISO or null",
    "duration_minutes": "integer or null"
  }
}

6. `delete_event`
Use only when the event must truly be deleted.
Arguments:
{
  "event_id": "string"
}

OBSERVATION CONVENTIONS:
- The gateway gives you normalized observation objects for reasoning. Do not expect raw Google Calendar API resources.
- `find_event` and `list_events` observations usually contain:
  {
    "items": [
      {
        "id": "...",
        "summary": "...",
        "start": "ISO",
        "end": "ISO",
        "status": "confirmed or similar"
      }
    ],
    "total_found": 1
  }
- `get_free_slots` observations usually contain:
  {
    "slots": [{"start": "ISO", "end": "ISO"}],
    "ranked_slots": [{"start": "ISO", "end": "ISO"}]
  }
- `create_event` and `update_event` observations may contain:
  {
    "event": {
      "id": "...",
      "summary": "...",
      "start": "ISO",
      "end": "ISO"
    }
  }
- `delete_event` observations may contain:
  {
    "deleted_event_id": "..."
  }

DECISION MODES:

1. `tool_call`
Use when the gateway must execute one tool next.
Return exactly one tool call, not a multi-step plan.

2. `clarify`
Use when critical information is still missing even after reading the current state.
Return a short Russian clarification question to the user.
No tool call.

3. `finish`
Use when you already have enough information in state and observations to answer the user or end the task.
Return a short Russian final message.
No tool call.
You may also return a structured `response_payload` when useful.

WHEN TO FINISH VS CALL A TOOL:
- If the user asks general chat or task splitting, usually finish immediately.
- If the user asks to create/update/delete something and all required identifiers/details are already known, call the write tool.
- If the user asks for schedule summary and you do not have events yet, call `list_events`.
- If the user asks to find a slot and you do not have availability yet, call `get_free_slots`.
- If observations already contain enough slot/event data to answer, finish.
- If observations already contain enough data to perform the next calendar mutation, call the mutation tool.

IMPORTANT SAFETY RULES:
- Prefer `update_event` over delete+create when simple modification is enough.
- For swapping two events, usually:
  1. find the first event
  2. find the second event
  3. update one
  4. update the other
  Do not delete both events unless deletion is explicitly required.
- Do not hallucinate event IDs. Use only IDs that appear in state or observations.
- Do not invent slots or events that are not present in observations.
- Return exactly one current next step. Do not output a long executable text plan.

STATE PATCH RULES:
- `state_patch` is optional.
- Use it to update `working_state.goal`, `working_state.status`, `working_state.plan_summary`,
  `working_state.pending_action`, or `working_state.resolved_entities`.
- Keep patches small and useful.

OUTPUT FORMAT (JSON ONLY, NO EXTRA TEXT):
{
  "response_type": "tool_call" | "clarify" | "finish",
  "assistant_message": "string or null",
  "response_payload": { ... optional structured data ... },
  "next_gateway_action": {
    "type": "none" | "tool_call",
    "tool_call": {
      "tool_name": "find_event" | "list_events" | "get_free_slots" | "create_event" | "update_event" | "delete_event",
      "arguments": { ... }
    } or null
  },
  "state_patch": { ... }
}
""".strip()

def make_system_prompt(current_time: str, timezone: str) -> str:
    return (
        STEP_SYSTEM_PROMPT_TEMPLATE
        .replace("__PERSONA__", CHAT_PERSONA)
        .replace("__CURRENT_TIME__", current_time)
        .replace("__TIMEZONE__", timezone)
    )



def load_dataset_samples(path: str) -> List[Dict[str, Any]]:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Dataset not found: {path_obj}")

    text = path_obj.read_text(encoding="utf-8-sig").strip()
    if not text:
        raise ValueError(f"Dataset is empty: {path_obj}")

    raw = json.loads(text)
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict) and isinstance(raw.get("samples"), list):
        return raw["samples"]

    raise ValueError("Dataset must be a JSON array or object with field 'samples'")

def normalize_step_payload(expected: Dict[str, Any]) -> Dict[str, Any]:
    data = dict(expected or {})

    if not isinstance(data.get("response_payload"), dict):
        data["response_payload"] = {}

    if not isinstance(data.get("state_patch"), dict):
        data["state_patch"] = {}

    if "next_gateway_action" not in data:
        root_tool_call = data.get("tool_call")
        if isinstance(root_tool_call, dict):
            data["next_gateway_action"] = {"type": "tool_call", "tool_call": root_tool_call}
        else:
            data["next_gateway_action"] = {"type": "none", "tool_call": None}

    gateway_action = data.get("next_gateway_action")
    if isinstance(gateway_action, dict):
        if "type" not in gateway_action:
            gateway_action["type"] = "tool_call" if gateway_action.get("tool_call") else "none"
        if "tool_call" not in gateway_action:
            gateway_action["tool_call"] = None
        data["next_gateway_action"] = gateway_action
    else:
        data["next_gateway_action"] = {"type": "none", "tool_call": None}

    if "response_type" not in data:
        if data["next_gateway_action"].get("type") == "tool_call":
            data["response_type"] = "tool_call"
        elif data.get("assistant_message"):
            data["response_type"] = "finish"
        else:
            data["response_type"] = "clarify"

    if "assistant_message" not in data:
        data["assistant_message"] = None

    return data

def normalize_train_cases(samples: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    cases: List[Dict[str, Any]] = []

    for sample_idx, sample in enumerate(samples):
        steps = sample.get("steps", [])
        if not isinstance(steps, list):
            continue

        for step_idx, step in enumerate(steps, start=1):
            request = step.get("step_request")
            expected = step.get("expected_step_response")

            if not isinstance(request, dict) or not isinstance(expected, dict):
                continue

            cases.append(
                {
                    "case_id": f"{sample.get('episode_id', f'ep_{sample_idx}')}:{step_idx}",
                    "user_text": request.get("text", ""),
                    "context": request.get("context", {}),
                    "state": request.get("state", {}),
                    "conversation": request.get("conversation"),
                    "all_context": request.get("all_context"),
                    "expected_json": normalize_step_payload(expected),
                }
            )

    return cases


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


BOS = "<|begin_of_text|>"
SYS_H = "<|start_header_id|>system<|end_header_id|>\n\n"
USR_H = "<|start_header_id|>user<|end_header_id|>\n\n"
AST_H = "<|start_header_id|>assistant<|end_header_id|>\n\n"
EOT = "<|eot_id|>"


STATE_PACKING_STAGES = [
    {"messages": 12, "actions": 10, "observations": 10, "items": 6, "slots": 6, "msg_chars": 320, "summary_chars": 400, "include_all_context": True},
    {"messages": 10, "actions": 8,  "observations": 8,  "items": 5, "slots": 5, "msg_chars": 280, "summary_chars": 320, "include_all_context": False},
    {"messages": 8,  "actions": 6,  "observations": 6,  "items": 4, "slots": 4, "msg_chars": 240, "summary_chars": 240, "include_all_context": False},
    {"messages": 6,  "actions": 5,  "observations": 5,  "items": 3, "slots": 3, "msg_chars": 200, "summary_chars": 180, "include_all_context": False},
    {"messages": 4,  "actions": 4,  "observations": 4,  "items": 2, "slots": 2, "msg_chars": 160, "summary_chars": 120, "include_all_context": False},
    {"messages": 3,  "actions": 3,  "observations": 3,  "items": 2, "slots": 2, "msg_chars": 120, "summary_chars": 80,  "include_all_context": False},
    {"messages": 2,  "actions": 2,  "observations": 2,  "items": 1, "slots": 1, "msg_chars": 100, "summary_chars": 0,   "include_all_context": False},
    {"messages": 0,  "actions": 1,  "observations": 1,  "items": 1, "slots": 1, "msg_chars": 0,   "summary_chars": 0,   "include_all_context": False},
]

def clip_text(text: Any, max_chars: int) -> Any:
    if not isinstance(text, str):
        return text
    if max_chars <= 0:
        return ""
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 1].rstrip() + "…"

def compact_action(action: Dict[str, Any]) -> Dict[str, Any]:
    tool_name = action.get("tool_name")
    args = action.get("arguments", {})
    compact = {"tool_name": tool_name, "arguments": args}
    return compact

def compact_observation(observation: Dict[str, Any], items_limit: int, slots_limit: int) -> Dict[str, Any]:
    tool_name = observation.get("tool_name")
    result = observation.get("result", {})

    compact_result = dict(result)

    if isinstance(compact_result.get("items"), list):
        compact_result["items"] = compact_result["items"][:items_limit]

    if isinstance(compact_result.get("slots"), list):
        compact_result["slots"] = compact_result["slots"][:slots_limit]

    if isinstance(compact_result.get("ranked_slots"), list):
        compact_result["ranked_slots"] = compact_result["ranked_slots"][:slots_limit]

    return {
        "tool_name": tool_name,
        "result": compact_result,
    }

def compact_state(state: Dict[str, Any], cfg: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(state, dict):
        return {}

    messages = state.get("messages", [])
    completed_actions = state.get("completed_actions", [])
    tool_observations = state.get("tool_observations", [])
    working_state = state.get("working_state", {})
    memory_summary = state.get("memory_summary", "")

    compact_messages = []
    if cfg["messages"] > 0 and isinstance(messages, list):
        for msg in messages[-cfg["messages"]:]:
            if not isinstance(msg, dict):
                continue
            compact_messages.append(
                {
                    "role": msg.get("role"),
                    "text": clip_text(msg.get("text", ""), cfg["msg_chars"]),
                }
            )

    compact_actions = []
    if cfg["actions"] > 0 and isinstance(completed_actions, list):
        compact_actions = [compact_action(a) for a in completed_actions[-cfg["actions"]:]]
        compact_actions = [a for a in compact_actions if isinstance(a, dict)]

    compact_observations = []
    if cfg["observations"] > 0 and isinstance(tool_observations, list):
        compact_observations = [
            compact_observation(o, cfg["items"], cfg["slots"])
            for o in tool_observations[-cfg["observations"]:]
            if isinstance(o, dict)
        ]

    compact_summary = clip_text(memory_summary, cfg["summary_chars"]) if cfg["summary_chars"] > 0 else ""

    return {
        "messages": compact_messages,
        "completed_actions": compact_actions,
        "tool_observations": compact_observations,
        "working_state": working_state if isinstance(working_state, dict) else {},
        "memory_summary": compact_summary,
    }

def build_user_payload(sample: Dict[str, Any], cfg: Dict[str, Any]) -> Dict[str, Any]:
    payload = {
        "text": sample.get("user_text", ""),
        "context": sample.get("context", {}),
        "state": compact_state(sample.get("state", {}), cfg),
    }

    if sample.get("conversation") is not None:
        payload["conversation"] = sample.get("conversation")

    if cfg["include_all_context"] and sample.get("all_context"):
        payload["all_context"] = sample.get("all_context")

    return payload

def build_prompt_and_target(sample: Dict[str, Any], cfg: Dict[str, Any]) -> Dict[str, str]:
    context = sample.get("context", {}) if isinstance(sample.get("context"), dict) else {}
    current_time = context.get("current_time", "")
    timezone = context.get("timezone", "UTC")

    system_prompt = make_system_prompt(current_time, timezone)
    user_payload = build_user_payload(sample, cfg)

    prompt_text = (
        f"{BOS}{SYS_H}{system_prompt}{EOT}"
        f"{USR_H}{json.dumps(user_payload, ensure_ascii=False)}{EOT}"
        f"{AST_H}"
    )

    target_text = f"{json.dumps(sample['expected_json'], ensure_ascii=False)}{EOT}"

    return {
        "prompt_text": prompt_text,
        "target_text": target_text,
    }

def find_best_pack(sample: Dict[str, Any], max_seq_length: int) -> Dict[str, Any]:
    best = None

    for cfg in STATE_PACKING_STAGES:
        texts = build_prompt_and_target(sample, cfg)

        prompt_ids = tokenizer(texts["prompt_text"], add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(texts["target_text"], add_special_tokens=False)["input_ids"]

        full_len = len(prompt_ids) + len(target_ids)

        candidate = {
            "cfg": cfg,
            "prompt_text": texts["prompt_text"],
            "target_text": texts["target_text"],
            "prompt_ids": prompt_ids,
            "target_ids": target_ids,
            "prompt_len": len(prompt_ids),
            "target_len": len(target_ids),
            "full_len": full_len,
            "fits": full_len <= max_seq_length,
        }

        best = candidate
        if candidate["fits"]:
            return candidate

    return best


raw_samples = load_dataset_samples(TRAIN_JSON_PATH)
train_cases = normalize_train_cases(raw_samples)

if not train_cases:
    raise ValueError("После нормализации не осталось ни одного train-case")

packed_stats = [find_best_pack(s, MAX_SEQ_LENGTH) for s in train_cases]

def percentile(sorted_vals, q):
    idx = int((len(sorted_vals) - 1) * q)
    return sorted_vals[idx]

prompt_lens = sorted(x["prompt_len"] for x in packed_stats)
target_lens = sorted(x["target_len"] for x in packed_stats)
full_lens = sorted(x["full_len"] for x in packed_stats)

fit_rate = sum(1 for x in packed_stats if x["fits"]) / len(packed_stats)

print("=" * 80)
print("СТАТИСТИКА ДЛИН ПОСЛЕ SMART PACKING")
print("=" * 80)
print(f"Train cases: {len(train_cases)}")
print(f"MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}")
print()
print(f"prompt_len: p50={percentile(prompt_lens, 0.5)}, p90={percentile(prompt_lens, 0.9)}, p95={percentile(prompt_lens, 0.95)}, max={max(prompt_lens)}")
print(f"target_len: p50={percentile(target_lens, 0.5)}, p90={percentile(target_lens, 0.9)}, p95={percentile(target_lens, 0.95)}, max={max(target_lens)}")
print(f"full_len:   p50={percentile(full_lens, 0.5)}, p90={percentile(full_lens, 0.9)}, p95={percentile(full_lens, 0.95)}, max={max(full_lens)}")
print()
print(f"fit_rate = {fit_rate:.4f}")
if fit_rate < 1.0:
    print("ВНИМАНИЕ: часть примеров всё ещё не помещается даже после smart packing.")
print("=" * 80)



def encode_sample(sample: Dict[str, Any]) -> Dict[str, List[int]]:
    packed = find_best_pack(sample, MAX_SEQ_LENGTH)

    prompt_ids = packed["prompt_ids"]
    target_ids = packed["target_ids"]

    total_len = len(prompt_ids) + len(target_ids)

    if total_len > MAX_SEQ_LENGTH:
        raise ValueError(
            f"Sample {sample.get('case_id')} still does not fit after smart packing: "
            f"{total_len} > {MAX_SEQ_LENGTH}"
        )

    input_ids = prompt_ids + target_ids
    attention_mask = [1] * len(input_ids)
    labels = ([-100] * len(prompt_ids)) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

encoded_train = [encode_sample(s) for s in train_cases]
train_ds = Dataset.from_list(encoded_train)


assert torch.cuda.is_available(), "Colab сейчас без GPU"
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
use_bf16 = major >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"GPU: {gpu_name}")
print(f"Compute dtype: {compute_dtype}")
print(f"Training max_seq_length: {MAX_SEQ_LENGTH}")
print(f"Encoded train cases: {len(encoded_train)}")


gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

for param in model.parameters():
    param.requires_grad = False

for name, module in model.named_modules():
    if "norm" in name.lower():
        try:
            module.to(torch.float32)
        except Exception:
            pass

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def make_inputs_require_grad(module, inputs, output):
        output.requires_grad_(True)
    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params:,} | "
        f"all params: {all_param:,} | "
        f"trainable%: {100 * trainable_params / all_param:.4f}"
    )

print_trainable_parameters(model)



def causal_lm_data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    pad_id = tokenizer.pad_token_id

    for f in features:
        pad_len = max_len - len(f["input_ids"])

        input_ids.append(f["input_ids"] + [pad_id] * pad_len)
        attention_mask.append(f["attention_mask"] + [0] * pad_len)
        labels.append(f["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }



class ProgressCallback(TrainerCallback):
    def __init__(self):
        self.last_reported_percent = -1

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"Training started: total_steps={state.max_steps}")

    def on_step_end(self, args, state, control, **kwargs):
        total_steps = state.max_steps or 0
        if total_steps <= 0:
            return control
        current_percent = int((state.global_step * 100) / total_steps)
        if current_percent > self.last_reported_percent:
            print(f"Training progress: {current_percent}% ({state.global_step}/{total_steps})")
            self.last_reported_percent = current_percent
        return control



training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_strategy="no",
    eval_strategy="no",
    do_eval=False,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none",
    seed=SEED,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_grad_norm=0.3,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    skip_memory_metrics=True,
)



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=causal_lm_data_collator,
    callbacks=[ProgressCallback()],
)



gc.collect()
torch.cuda.empty_cache()

trainer.train()

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

run_info = {
    "base_model": BASE_MODEL,
    "output_dir": OUTPUT_DIR,
    "max_steps": MAX_STEPS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACC_STEPS,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "max_seq_length": MAX_SEQ_LENGTH,
    "fit_rate_after_packing": fit_rate,
}
(output_path / "run_info.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Готово.")
print(f"Adapter saved to: {OUTPUT_DIR}")


СТАТИСТИКА ДЛИН ПОСЛЕ SMART PACKING
Train cases: 576
MAX_SEQ_LENGTH = 3500

prompt_len: p50=1928, p90=2154, p95=2269, max=2336
target_len: p50=98, p90=116, p95=118, max=121
full_len:   p50=1995, p90=2253, p95=2350, max=2405

fit_rate = 1.0000
GPU: Tesla T4
Compute dtype: torch.float16
Training max_seq_length: 3500
Encoded train cases: 576


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 41,943,040 | all params: 4,582,543,360 | trainable%: 0.9153
Training started: total_steps=144
Training progress: 0% (1/144)


Step,Training Loss
5,0.326593


Training progress: 1% (2/144)
Training progress: 2% (3/144)
Training progress: 3% (5/144)
Training progress: 4% (6/144)
Training progress: 5% (8/144)
Training progress: 6% (9/144)


Step,Training Loss
5,0.326593
10,0.114182
15,0.076794
20,0.043034
25,0.029858
30,0.021243
35,0.021821
40,0.011153
45,0.004821
50,0.006505


Training progress: 7% (11/144)
Training progress: 8% (12/144)
Training progress: 9% (13/144)
Training progress: 10% (15/144)
Training progress: 11% (16/144)
Training progress: 12% (18/144)
Training progress: 13% (19/144)
Training progress: 14% (21/144)
Training progress: 15% (22/144)
Training progress: 16% (24/144)
Training progress: 17% (25/144)
Training progress: 18% (26/144)
Training progress: 19% (28/144)
Training progress: 20% (29/144)
Training progress: 21% (31/144)
Training progress: 22% (32/144)
Training progress: 23% (34/144)
Training progress: 24% (35/144)
Training progress: 25% (36/144)
Training progress: 26% (38/144)
Training progress: 27% (39/144)
Training progress: 28% (41/144)
Training progress: 29% (42/144)
Training progress: 30% (44/144)
Training progress: 31% (45/144)
Training progress: 32% (47/144)
Training progress: 33% (48/144)
Training progress: 34% (49/144)
Training progress: 35% (51/144)
Training progress: 36% (52/144)
Training progress: 37% (54/144)
Training pr